In [14]:
import os
import psycopg2 as psql
from pydantic import BaseModel
from typing import Optional, List
import sys
import os
import numpy as np
from sentence_transformers import SentenceTransformer,CrossEncoder

import gc
import torch

db_database = os.getenv("DB_DATABASE")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")

DB_CONFIG = {
    "dbname": db_database,
    "user": db_user,
    "password": db_password,
    "host": db_host
}

In [15]:
class EmbeddingChunk(BaseModel):
    id:int
    text:str
    pages:List[int]
    token_count:int
    embedding:List[float]
    document:str
    similarity:float


def retrieve_embeddings():
    """
    Retrieves all embeddings from the database.
    """
    
    
    conn = psql.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    # Fetch all embeddings from the database
    cur.execute("SELECT id, text, pages, token_count, embedding, embeddings_model, document FROM embeddings_table_v2;")
    results = cur.fetchall()
    cur.close()
    conn.close()
    
    return results


def convert_chunk_to_json(embedding_tuple):
    """
    Converts a chunk of embeddings to a JSON object.
    """
    
    return EmbeddingChunk(
        id=embedding_tuple[0][0],
        text=embedding_tuple[0][1],
        pages=embedding_tuple[0][2],
        token_count=embedding_tuple[0][3],
        embedding=embedding_tuple[0][4],
        embeddings_model=embedding_tuple[0][5],
        document=embedding_tuple[0][6],
        similarity=embedding_tuple[1]
    ).model_dump()
    
    
def semantic_search(user_input,db_embeddings, top_k, model):
    """
    Performs semantic search on the database of embeddings.
    """
    
    user_embedding = model.encode(user_input)
    
    # Calculate cosine similarity between user input and all embeddings
    similarities = []
    for db_embedding in db_embeddings:
        db_embedding_vector = np.array(db_embedding[4])
        similarity = np.dot(user_embedding, db_embedding_vector) / (np.linalg.norm(user_embedding) * np.linalg.norm(db_embedding_vector))
        similarities.append((db_embedding, similarity))
        
        
    # Sort by similarity
    similarities.sort(key=lambda x: x[1], reverse=True)
    similarities = [convert_chunk_to_json(similarity) for similarity in similarities]
    #similarities = process_similarities(similarities)
    
    # Return top_p results
    return similarities[:top_k]

def softmax(x):
    """Computes softmax for an array of scores."""
    exp_x = np.exp(x - np.max(x))  # Subtract max score for numerical stability
    return exp_x / exp_x.sum()

def process_context(user_input,selected_chunks,cross_encoder):
    """
    Processes context to be fed into the LLM.
    """
    results = cross_encoder.predict([[user_input, chunk['text']] for chunk in selected_chunks])/200
    softmax_scores = softmax(results)

    threshold = 0.01

    filtered_chunks = [selected_chunks[i] for i, score in enumerate(softmax_scores) if score > threshold]

    context = " ".join([chunk['text'] for chunk in filtered_chunks])
    
    return context


def generate_rag(model, system_prompt, user_prompt, top_k, embeddings_model_id, cross_encoder_id):
    """
    Generates a response using the RAG pipeline.
    """
    
    print("Retrieving embeddings...")
    db_embeddings = retrieve_embeddings()
    embeddings_model = SentenceTransformer(embeddings_model_id)
    print("Performing semantic search...")
    selected_chunks = semantic_search(user_prompt, db_embeddings, top_k, embeddings_model)
    
    print("Unloading ambeddings model...")
    del embeddings_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
                
    print("Processing context...")
    cross_encoder = CrossEncoder(cross_encoder_id)
    context = process_context(user_prompt,selected_chunks,cross_encoder)
    print("Processing references...")
    document_pages = get_references(selected_chunks)
    prompt = f"Based only on the following in markdown: {context} \nAnswer this: {user_prompt}"
    print("Calling LLMP...")
    response = llmp_call(prompt, system_prompt, model)
    
    return response,document_pages
    print(F"""{response['message']['content']}\n
      documents: {documents}\n
      pages: {unique_pages}""")
    
def get_references(selected_chunks):
    document_pages = {}

    for chunk in selected_chunks:
        doc = chunk['document']
        if doc not in document_pages:
            document_pages[doc] = set()  # Use a set to ensure uniqueness

        document_pages[doc].update(chunk["pages"])

    # Convert sets to sorted lists for better readability
    document_pages = {doc: sorted(pages) for doc, pages in document_pages.items()}

    return document_pages

In [16]:
# ASGENT 0 SIDE NOT RAG
from typing import Dict
import requests

llmp_url = os.getenv("LLMP_URL")
llmp_password = os.getenv("LLMP_PASSWORD")


class GenerateRequest(BaseModel):
    model: str
    system_prompt: str = ''
    prompt: str
    format: Optional[dict] = None
    image: Optional[str] = None
    tools: Optional[List[Dict]] = None
    src: str = None

def llmp_call(prompt, system_prompt, model):
        """ 
        Call the LLMP API to generate a response
        All related to the call is processed here
        """
        
        headers = {
        "Content-Type": "application/json",
        "Authorization": llmp_password
    }
        
        # Construct request payload
        request_data = GenerateRequest(
        model=model,
        system_prompt=system_prompt,
        prompt=prompt,
        tools=None,
        src="RAG test")
        
        payload = request_data.model_dump(exclude_none=True)

        try:
            response = requests.post(llmp_url, headers=headers, json=payload)
            response.raise_for_status()  # Raise an error for bad responses (4xx, 5xx)
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            return None

In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)
    
    
from src.pipelines.rag_pipeline import generate_rag

model = "llama3.2:latest"
system_prompt = "You are an assistant. Your provide answers based on provided text."
user_prompt = "What is enthalpy?"
top_k = 5
embeddings_model_id = "intfloat/e5-large-v2"
cross_encoder_id = "cross-encoder/ms-marco-MiniLM-L-6-v2"

rag_output = generate_rag(model, system_prompt, user_prompt, top_k, embeddings_model_id,cross_encoder_id)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


ModuleNotFoundError: No module named 'utils'

In [19]:
rag_output[0]

{'model': 'llama3.2:latest',
 'created_at': '2025-03-04T08:05:56.5255821Z',
 'message': {'role': 'assistant',
  'content': 'Enthalpy (H) is defined as:\n\nF = U - TS\n\nwhere F is the Helmholtz function, U is the internal energy, T is the temperature, and S is the entropy.'},
 'done_reason': 'stop',
 'done': True,
 'total_duration': 9311805600,
 'load_duration': 6.79,
 'prompt_eval_count': 1890,
 'prompt_eval_duration': 1.31,
 'eval_count': 44,
 'eval_duration': 0.73,
 'system_prompt': 'You are an assistant. Your provide answers based on provided text.',
 'prompt': 'Based only on the following in markdown: the system at con- stant pressure, the enthalpy Hof the system goes up. If heat is provided bythe system to its surroundings Hgoes down.ditions are relatively easy to obtain: an experiment which is open to the air in a laboratory is usually at constant pressure since pressure is provided by the atmosphere.4We also conclude from eqn 16.9 that if 4At a given latitude, the atmosphere pr

In [21]:
print(rag_output[1])

{'Concepts in Thermal Physics - S. Blundell, K. Blundell (Oxford, 2006) WW.pdf': [182, 183, 184, 185, 188, 189], 'Electricity and Magnetism - E. Purcel, D. Morin.pdf': [184, 185, 188, 189]}
